# Machine Learning Assignment 2 - Complete Solution
## Cell-by-Cell Implementation Guide

**Dataset**: We'll use the **Heart Disease UCI Dataset** (meets requirements: 13+ features, 1000+ instances)

---

## Cell 1: Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

# Model persistence
import pickle
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## Cell 2: Load Dataset
**TODO**: Replace this with your chosen dataset from Kaggle/UCI

In [ ]:
# Example: Heart Disease Dataset from UCI
# Download from: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

# Load your dataset
df = pd.read_csv('heart.csv')  # Replace with your dataset path

print(f"Dataset Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

## Cell 3: Data Exploration

In [ ]:
# Check dataset info
print("Dataset Information:")
print(df.info())

print("\n" + "="*50)
print("Statistical Summary:")
print(df.describe())

print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum())

print("\n" + "="*50)
print("Target Variable Distribution:")
print(df.iloc[:, -1].value_counts())  # Assuming last column is target

## Cell 4: Data Preprocessing

In [ ]:
# Separate features and target
# Adjust column name based on your dataset
X = df.drop('target', axis=1)  # Replace 'target' with your target column name
y = df['target']  # Replace 'target' with your target column name

# Handle missing values if any
X = X.fillna(X.mean())

# Split data into train and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling (important for some models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler for later use
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## Cell 5: Define Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a model and return all required metrics
    """
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Get probability predictions for AUC (if available)
    try:
        if hasattr(model, 'predict_proba'):
            y_pred_proba = model.predict_proba(X_test)
            # For binary classification
            if len(np.unique(y_test)) == 2:
                auc = roc_auc_score(y_test, y_pred_proba[:, 1])
            # For multi-class classification
            else:
                auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
        else:
            auc = 'N/A'
    except:
        auc = 'N/A'
    
    # Calculate metrics
    metrics = {
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(auc, 4) if auc != 'N/A' else 'N/A',
        'Precision': round(precision_score(y_test, y_pred, average='weighted'), 4),
        'Recall': round(recall_score(y_test, y_pred, average='weighted'), 4),
        'F1': round(f1_score(y_test, y_pred, average='weighted'), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4)
    }
    
    return metrics

print("Evaluation function defined successfully!")

## Cell 6: Model 1 - Logistic Regression

In [ ]:
print("Training Logistic Regression...")

# Initialize and train model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Evaluate model
lr_metrics = evaluate_model(lr_model, X_test_scaled, y_test, 'Logistic Regression')

# Save model
with open('model/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

print("Logistic Regression - Metrics:")
for key, value in lr_metrics.items():
    print(f"{key}: {value}")

## Cell 7: Model 2 - Decision Tree Classifier

In [ ]:
print("Training Decision Tree...")

# Initialize and train model
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_model.fit(X_train, y_train)  # Decision trees don't require scaling

# Evaluate model
dt_metrics = evaluate_model(dt_model, X_test, y_test, 'Decision Tree')

# Save model
with open('model/decision_tree.pkl', 'wb') as f:
    pickle.dump(dt_model, f)

print("Decision Tree - Metrics:")
for key, value in dt_metrics.items():
    print(f"{key}: {value}")

## Cell 8: Model 3 - K-Nearest Neighbors (kNN)

In [ ]:
print("Training K-Nearest Neighbors...")

# Initialize and train model
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

# Evaluate model
knn_metrics = evaluate_model(knn_model, X_test_scaled, y_test, 'kNN')

# Save model
with open('model/knn.pkl', 'wb') as f:
    pickle.dump(knn_model, f)

print("kNN - Metrics:")
for key, value in knn_metrics.items():
    print(f"{key}: {value}")

## Cell 9: Model 4 - Naive Bayes (Gaussian)

In [ ]:
print("Training Naive Bayes (Gaussian)...")

# Initialize and train model
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Evaluate model
nb_metrics = evaluate_model(nb_model, X_test_scaled, y_test, 'Naive Bayes')

# Save model
with open('model/naive_bayes.pkl', 'wb') as f:
    pickle.dump(nb_model, f)

print("Naive Bayes - Metrics:")
for key, value in nb_metrics.items():
    print(f"{key}: {value}")

## Cell 10: Model 5 - Random Forest (Ensemble)

In [ ]:
print("Training Random Forest...")

# Initialize and train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Evaluate model
rf_metrics = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

# Save model
with open('model/random_forest.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("Random Forest - Metrics:")
for key, value in rf_metrics.items():
    print(f"{key}: {value}")

## Cell 11: Model 6 - XGBoost (Ensemble)

In [ ]:
print("Training XGBoost...")

# Initialize and train model
xgb_model = XGBClassifier(n_estimators=100, random_state=42, max_depth=6, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

# Evaluate model
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

# Save model
with open('model/xgboost.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

print("XGBoost - Metrics:")
for key, value in xgb_metrics.items():
    print(f"{key}: {value}")

## Cell 12: Compile All Results into Comparison Table

In [ ]:
# Compile all metrics
all_metrics = [
    lr_metrics,
    dt_metrics,
    knn_metrics,
    nb_metrics,
    rf_metrics,
    xgb_metrics
]

# Create DataFrame
results_df = pd.DataFrame(all_metrics)
results_df = results_df[['Model', 'Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']]

# Display results
print("\n" + "="*80)
print("MODEL COMPARISON TABLE")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

# Save to CSV
results_df.to_csv('model_comparison_results.csv', index=False)
print("\nResults saved to 'model_comparison_results.csv'")

## Cell 13: Visualize Model Performance

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = ['Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F']

for idx, (metric, ax) in enumerate(zip(metrics_to_plot, axes.flatten())):
    if metric in results_df.columns:
        # Handle 'N/A' values in AUC
        plot_data = results_df[[metric]].copy()
        if metric == 'AUC':
            plot_data[metric] = pd.to_numeric(plot_data[metric], errors='coerce')
        
        ax.barh(results_df['Model'], plot_data[metric], color=colors[idx], alpha=0.8)
        ax.set_xlabel(metric, fontweight='bold')
        ax.set_title(f'{metric} Comparison', fontweight='bold')
        ax.set_xlim(0, 1.0)
        
        # Add value labels
        for i, v in enumerate(plot_data[metric]):
            if pd.notna(v):
                ax.text(v + 0.02, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'model_comparison_visualization.png'")

## Cell 14: Generate Confusion Matrices for Best Model

In [ ]:
# Find best model based on accuracy
best_model_name = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
print(f"Best performing model: {best_model_name}")

# Load best model and generate confusion matrix
model_mapping = {
    'Logistic Regression': (lr_model, X_test_scaled),
    'Decision Tree': (dt_model, X_test),
    'kNN': (knn_model, X_test_scaled),
    'Naive Bayes': (nb_model, X_test_scaled),
    'Random Forest': (rf_model, X_test),
    'XGBoost': (xgb_model, X_test)
}

best_model, X_test_for_model = model_mapping[best_model_name]
y_pred = best_model.predict(X_test_for_model)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title(f'Confusion Matrix - {best_model_name}', fontweight='bold', fontsize=14)
plt.ylabel('Actual', fontweight='bold')
plt.xlabel('Predicted', fontweight='bold')
plt.savefig('confusion_matrix_best_model.png', dpi=300, bbox_inches='tight')
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Cell 15: Save Test Data for Streamlit App

In [ ]:
# Save test data with predictions for Streamlit demo
test_data_with_predictions = X_test.copy()
test_data_with_predictions['Actual'] = y_test.values

# Save a small sample for upload demo (Streamlit free tier limitation)
sample_test_data = test_data_with_predictions.sample(n=min(50, len(test_data_with_predictions)), random_state=42)
sample_test_data.to_csv('sample_test_data.csv', index=False)

print(f"Sample test data saved: {sample_test_data.shape}")
print("File: sample_test_data.csv")

## ✅ ASSIGNMENT COMPLETION CHECKLIST

### What You've Accomplished:
1. ✅ Implemented 6 classification models
2. ✅ Calculated all 6 required metrics for each model
3. ✅ Created comparison table
4. ✅ Generated visualizations
5. ✅ Saved all models for deployment

### Next Steps:
1. Create Streamlit app (app.py)
2. Create requirements.txt
3. Write README.md with observations
4. Push to GitHub
5. Deploy on Streamlit Cloud
6. Take screenshot on BITS Virtual Lab
7. Create submission PDF